In [1]:
!pip install -U transformers datasets accelerate sentencepiece evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 110.2 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.3
    Uninstalling datasets-4.8.3:
      Successfully uninstalled datasets-4.8.3
  Attempting uninstal

Step 2: Import Libraries and Download Dataset

In [2]:
import kagglehub
import pandas as pd
import numpy as np
import torch
import transformers

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

from datasets import Dataset

import warnings
import os

warnings.filterwarnings('ignore')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# GPU Check
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

print("Transformers Version:", transformers.__version__)

# Download dataset
print("\nDownloading Sandhi dataset...")

path = kagglehub.dataset_download("tanujsaxena/sandhi-data")

print("Path to dataset files:", path)

# Dataset files
files = os.listdir(path)

print(f"Files in dataset: {files}")

GPU Available: True
GPU Name: Tesla T4
Transformers Version: 5.8.0

Path to dataset files: /kaggle/input/datasets/tanujsaxena/sandhi-data
Files in dataset: ['sandhi_data.xlsx']


Step 3: Load Excel File

In [3]:
# Load Excel file
excel_file = os.path.join(path, 'sandhi_data.xlsx')

print(f"\nLoading Excel file: {excel_file}")

# Available sheets
xl_file = pd.ExcelFile(excel_file)

print(f"Available sheets: {xl_file.sheet_names}")

# Load first sheet
df = pd.read_excel(excel_file, sheet_name=0)

print("\nDataset Preview:")
display(df.head(10))

print(f"\nDataset Shape: {df.shape}")

print(f"\nColumns:")
print(df.columns.tolist())

print(f"\nData Types:")
print(df.dtypes)

print(f"\nNull Values:")
print(df.isnull().sum())

print(f"\nDuplicate Rows: {df.duplicated().sum()}")


Loading Excel file: /kaggle/input/datasets/tanujsaxena/sandhi-data/sandhi_data.xlsx
Available sheets: ['Sheet1']

Dataset Preview:


,Word,Split
0,प्रथमोऽङ्कः,प्रथमः+अङ्कः
1,शब्द इव,शब्दः+इव
2,इत इतः,इतः+इतः
3,कुतो नु,कुतः+नु
4,खल्वेष समुत्थितो समुत्थितो ध्वनिः,खलु+एषः+समुत्थितः+समुत्थितः+ध्वनिः
5,एष खलु,एषः+खलु
6,रामो हनुमांश्च,रामः+हनुमान्+च
7,भयं त्यज,भयम्+त्यज
8,ममापि,मम+अपि
9,कथं कथं कथं सुग्रीव सुग्रीव इति,कथम्+कथम्+कथम्+सुग्रीवः+सुग्रीवः+इति



Dataset Shape: (13930, 2)

Columns:
['Word', 'Split']

Data Types:
Word     object
Split    object
dtype: object

Null Values:
Word     0
Split    5
dtype: int64

Duplicate Rows: 215


Analyze Dataset Quality First

In [4]:
# Display column names
print("Column names:", df.columns.tolist())

# Rename columns
if 'Compound' in df.columns:
    df = df.rename(columns={
        'Compound': 'input_text',
        'Word1': 'target_text'
    })

else:
    col1, col2 = df.columns[0], df.columns[1]

    print(f"\nUsing columns:")
    print(f"Input Column  : {col1}")
    print(f"Target Column : {col2}")

    df = df.rename(columns={
        col1: 'input_text',
        col2: 'target_text'
    })

# Keep required columns
df = df[['input_text', 'target_text']].copy()

# Remove duplicates
df = df.drop_duplicates()

# Remove nulls
df = df.dropna(subset=['input_text', 'target_text'])

# Convert to string
df['input_text'] = df['input_text'].astype(str)
df['target_text'] = df['target_text'].astype(str)

# Strip whitespace
df['input_text'] = df['input_text'].str.strip()
df['target_text'] = df['target_text'].str.strip()

# Remove empty rows
df = df[
    (df['input_text'].str.len() > 0) &
    (df['target_text'].str.len() > 0)
]

# Normalize Unicode
df['input_text'] = df['input_text'].str.normalize('NFKC')
df['target_text'] = df['target_text'].str.normalize('NFKC')

print(f"\nCleaned dataset shape: {df.shape}")

print("\nFirst 15 samples:")
display(df.head(15))

print("\nSample Statistics:")
print(f"Average input length  : {df['input_text'].str.len().mean():.2f}")
print(f"Average target length : {df['target_text'].str.len().mean():.2f}")

print(f"\nDuplicate samples remaining: {df.duplicated().sum()}")

Column names: ['Word', 'Split']

Using columns:
Input Column  : Word
Target Column : Split

Cleaned dataset shape: (13710, 2)

First 15 samples:


,input_text,target_text
0,प्रथमोऽङ्कः,प्रथमः+अङ्कः
1,शब्द इव,शब्दः+इव
2,इत इतः,इतः+इतः
3,कुतो नु,कुतः+नु
4,खल्वेष समुत्थितो समुत्थितो ध्वनिः,खलु+एषः+समुत्थितः+समुत्थितः+ध्वनिः
5,एष खलु,एषः+खलु
6,रामो हनुमांश्च,रामः+हनुमान्+च
7,भयं त्यज,भयम्+त्यज
8,ममापि,मम+अपि
9,कथं कथं कथं सुग्रीव सुग्रीव इति,कथम्+कथम्+कथम्+सुग्रीवः+सुग्रीवः+इति



Sample Statistics:
Average input length  : 12.67
Average target length : 14.01

Duplicate samples remaining: 17


CLEANING AND PREPARING DATA

In [5]:
# Convert to HF Dataset
dataset = Dataset.from_pandas(
    df[['input_text', 'target_text']]
)

# Train-validation split
train_test_split = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

print(f"Training samples   : {len(train_dataset)}")
print(f"Validation samples : {len(eval_dataset)}")

print("\nExample Training Samples:")

for i in range(3):

    print(f"\nExample {i+1}")

    print(f"Input : {train_dataset[i]['input_text']}")
    print(f"Target: {train_dataset[i]['target_text']}")

# Leakage check
train_inputs = set(train_dataset['input_text'])
eval_inputs = set(eval_dataset['input_text'])

overlap = train_inputs.intersection(eval_inputs)

print(f"\nOverlap between train and validation: {len(overlap)}")

Training samples   : 10968
Validation samples : 2742

Example Training Samples:

Example 1
Input : पृथग् भवन्ति
Target: पृथक्+भवन्ति

Example 2
Input : दुःखालयमशाश्वतम्
Target: दुःख+आलयम्+अशाश्वतम्

Example 3
Input : समर्थो नास्ति
Target: समर्थः+नास्ति

Overlap between train and validation: 8


In [6]:
from transformers import DataCollatorForSeq2Seq

# Model name
model_name = "ai4bharat/IndicBART"

print(f"\nLoading model: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Silence warning
model.config.tie_word_embeddings = False

# Move to GPU
if torch.cuda.is_available():
    model = model.cuda()

print("✓ Model loaded successfully")

print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

print(f"Model parameters: {model.num_parameters()/1e6:.2f}M")


Loading model: ai4bharat/IndicBART


config.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.90M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/221 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/976M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/976M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/267 [00:00<?, ?it/s]

✓ Model loaded successfully
Tokenizer vocab size: 64000
Model parameters: 244.02M


In [7]:
def preprocess_function(examples):

    # Add Sanskrit language tag
    inputs = [
        f"<2sa> {text}"
        for text in examples['input_text']
    ]

    targets = [
        f"<2sa> {text}"
        for text in examples['target_text']
    ]

    # Tokenize inputs
    model_inputs = tokenizer(
        inputs,
        max_length=128,
        truncation=True
    )

    # Tokenize targets
    labels = tokenizer(
        targets,
        max_length=128,
        truncation=True
    )

    label_ids = labels["input_ids"]

    # Ignore padding tokens in loss
    label_ids = [
        [
            (token if token != tokenizer.pad_token_id else -100)
            for token in label
        ]
        for label in label_ids
    ]

    model_inputs["labels"] = label_ids

    return model_inputs


print("\nTokenizing training dataset...")

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=32,
    remove_columns=['input_text', 'target_text']
)

print("Tokenizing validation dataset...")

tokenized_eval = eval_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=32,
    remove_columns=['input_text', 'target_text']
)

print("✓ Tokenization complete!")

print("\nSample Tokenized Example:")
print(tokenized_train[0].keys())


Tokenizing training dataset...


Map:   0%|          | 0/10968 [00:00<?, ? examples/s]

Tokenizing validation dataset...


Map:   0%|          | 0/2742 [00:00<?, ? examples/s]

✓ Tokenization complete!

Sample Tokenized Example:
dict_keys(['__index_level_0__', 'input_ids', 'attention_mask', 'labels'])


In [8]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(

    output_dir='/kaggle/working/sandhi_model',

    num_train_epochs=10,

    learning_rate=3e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    weight_decay=0.01,

    warmup_steps=100,

    logging_steps=50,

    eval_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=2,

    predict_with_generate=True,

    fp16=torch.cuda.is_available(),

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,

    seed=42,

    report_to="none"
)

In [9]:
from transformers import Seq2SeqTrainer

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

print("\n" + "=" * 60)
print("Starting training on Tesla T4 GPU...")
print("=" * 60 + "\n")

torch.cuda.empty_cache()

trainer.train()

print("\n" + "=" * 60)
print("✓ Training complete!")
print("=" * 60)


Starting training on Tesla T4 GPU...



[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss
1,2.682202,1.992350
2,1.738915,1.514004
3,1.539693,1.313428
4,1.300006,1.156663
5,1.183732,1.056143
6,1.096507,1.002914
7,1.097228,0.970907
8,1.088952,0.944360
9,1.110316,0.934037
10,1.079253,0.930026


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].



✓ Training complete!


In [10]:
import os

# Output directory
model_output_path = '/kaggle/working/sandhi_model_final'

os.makedirs(model_output_path, exist_ok=True)

# Save model
trainer.save_model(model_output_path)

# Save tokenizer
tokenizer.save_pretrained(model_output_path)

print(f"✓ Model saved to: {model_output_path}")

print("\nModel files saved:")

for file in os.listdir(model_output_path):
    print(f"  - {file}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Model saved to: /kaggle/working/sandhi_model_final

Model files saved:
  - training_args.bin
  - model.safetensors
  - tokenizer_config.json
  - tokenizer.json
  - config.json
  - generation_config.json


In [11]:
# Get trained model
trained_model = trainer.model
trained_tokenizer = tokenizer

# Move to GPU
if torch.cuda.is_available():
    trained_model = trained_model.cuda()

trained_model.eval()

print("\n" + "=" * 70)
print("Testing Model Predictions on Validation Data")
print("=" * 70)

# Random samples
test_count = min(15, len(eval_dataset))

test_indices = np.random.choice(
    len(eval_dataset),
    test_count,
    replace=False
)

correct_predictions = 0

for i, idx in enumerate(test_indices):

    sample = eval_dataset[int(idx)]

    input_text = sample['input_text']
    target_text = sample['target_text']

    # Add Sanskrit tag
    formatted_input = f"<2sa> {input_text}"

    # Tokenize
    inputs = trained_tokenizer(
        formatted_input,
        max_length=128,
        truncation=True,
        return_tensors="pt"
    )

    # Move tensors to GPU
    if torch.cuda.is_available():
        inputs = {
            k: v.cuda()
            for k, v in inputs.items()
        }

    # Generate
    with torch.no_grad():

        outputs = trained_model.generate(
            **inputs,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

    # Decode
    predicted_text = trained_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Clean output
    predicted_text = predicted_text.replace("<2sa>", "")
    predicted_text = " ".join(predicted_text.split()).strip()

    target_text = target_text.replace("<2sa>", "")
    target_text = " ".join(target_text.split()).strip()

    # Exact match
    is_correct = (
        predicted_text.strip() ==
        target_text.strip()
    )

    if is_correct:
        correct_predictions += 1

    status = "✓ CORRECT" if is_correct else "✗ WRONG"

    print(f"\n{i+1}. {status}")

    print(f"Input (Sandhi):")
    print(f"  {input_text}")

    print(f"Expected Split:")
    print(f"  {target_text}")

    print(f"Predicted Split:")
    print(f"  {predicted_text}")

# Accuracy
accuracy = 100 * correct_predictions / test_count

print("\n" + "=" * 70)

print(
    f"ACCURACY: "
    f"{correct_predictions}/{test_count} "
    f"({accuracy:.1f}%)"
)

print("=" * 70)


Testing Model Predictions on Validation Data

1. ✗ WRONG
Input (Sandhi):
  क्रोधाद्भवति
Expected Split:
  क्रोधात्+भवति
Predicted Split:
  <s> करध+अभवत

2. ✗ WRONG
Input (Sandhi):
  पश्याऽत्र
Expected Split:
  पश्य+अत्र
Predicted Split:
  <s> पशय+अतर2sa> पशय+अतर

3. ✗ WRONG
Input (Sandhi):
  यत्‍स्‍वामी
Expected Split:
  यत्+स्वामी
Predicted Split:
  <s> यत+सवम

4. ✗ WRONG
Input (Sandhi):
  सुपां सुलुक्पूर्वसवर्णाऽऽच्छेयाडाड्यायाजालः
Expected Split:
  सुपाम्+सुलुक्पूर्वसवर्णाऽऽच्छेयाडाड्यायाजालः
Predicted Split:
  <s> सप+अलकपरवसवरणऽऽचछयडडययजलअसमअसम+अचछयडडययजल2sa> सप+असलसरवसवसवसवरणअसमवरण+अचछयडडययजल

5. ✗ WRONG
Input (Sandhi):
  प्रतियोगिताऽन्वयः
Expected Split:
  प्रतियोगिता+अन्वयः
Predicted Split:
  <s> परतयगत+अनवय

6. ✗ WRONG
Input (Sandhi):
  प्राणायामपरायणाः
Expected Split:
  प्राण+आयामपर+अयनाः
Predicted Split:
  <s> परणयम+परयण

7. ✗ WRONG
Input (Sandhi):
  समुदोरजः पशुषु
Expected Split:
  समुदोः+अजः+पशुषु
Predicted Split:
  <s> समदरज+पशष2sa> समदरज+पशष2sa> समदरज+पशष

8. ✗ WRONG
Inp